# Module 2 — Exercises (Student Worksheet)
### Graphs, Message Passing, EdgeConv & ParticleNet · TIFR ML School 2026

Student worksheet for the six exercises at the end of
`Module2_Graphs_MessagePassing_ParticleNet.ipynb`. **Self-contained**: the *Setup* section re-defines the
k-NN graph builder, EdgeConv/ParticleNet/GravNet, and loads JetClass into PyG `Data` objects.

| # | Exercise | Idea it drives home |
|---|----------|---------------------|
| 1 | **k matters** | neighbourhood size is an accuracy/compute knob |
| 2 | **Static vs dynamic** | rebuilding the graph in learned space is what makes ParticleNet more than a fixed-graph GNN |
| 3 | **Aggregator** | `sum`/`mean`/`max` over *fixed-k* neighbours — and why that differs from Module-1 pooling |
| 4 | **Edge features** | does handing the net ΔR help, or does `x_j−x_i` already contain it? |
| 5 | **Train GravNet** | a *learned* neighbourhood space vs a recomputed k-NN |
| 6 | **PFN vs ParticleNet** | *where* on the ROC do edges actually pay off |

> **One design choice that pays off everywhere:** we make `ParticleNet` **configurable** (`k`, `aggr`,
> `dynamic`), so exercises 1–3 are just three different constructor calls on the same class. And a PFN over
> a PyG batch is literally `global_add_pool(φ(x), batch)` — so we can run the PFN on the **identical** graph
> splits used for ParticleNet (exercise 6's "same test jets", for free).
>
> **Speed knobs:** `N_PER_CLS` and `EPOCHS`. Defaults run in a few minutes on a laptop; raise to sharpen.

> **How to use this worksheet.** Run the **Setup** section as-is (it loads the data and provides the models/helpers from the module). Then complete each exercise where you see `# TODO` and `raise NotImplementedError`. Read the **prompt** above each exercise — it sketches the approach. Check your work against the matching `*_Exercises_Solutions.ipynb`.

## Setup — imports, graph machinery, data, models

In [ ]:
import os, time, random, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
import torch_geometric
from torch_geometric.nn import MessagePassing, global_mean_pool, global_add_pool
from torch_geometric.utils import scatter
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import roc_auc_score, roc_curve

torch.manual_seed(0); np.random.seed(0); random.seed(0)
device = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))
print("torch", torch.__version__, "| PyG", torch_geometric.__version__, "| device", device)

In [ ]:
# --- k-NN graph, EdgeConv, EdgeConvBlock (from Module 2) ---------------------------------
def knn_graph_native(x, k, batch):
    """Within-graph k-NN edge_index [2,E] = [source(j), target(i)]. Pure torch."""
    with torch.no_grad():
        N = x.size(0)
        x2 = (x*x).sum(1, keepdim=True)
        dist2 = x2 + x2.t() - 2.0*(x @ x.t())
        same = batch.unsqueeze(0) == batch.unsqueeze(1)
        dist2 = dist2.masked_fill(~same, float("inf")); dist2.fill_diagonal_(float("inf"))
        kk = min(k, max(N-1, 1))
        knn_d, idx = dist2.topk(kk, dim=1, largest=False)
        valid = torch.isfinite(knn_d)
        centre = torch.arange(N, device=x.device).unsqueeze(1).expand(-1, kk)
        return torch.stack([idx[valid], centre[valid]], dim=0)

class EdgeConv(MessagePassing):
    """x_i' = AGGR_j  MLP([x_i, x_j - x_i])."""
    def __init__(self, in_c, out_c, aggr="mean"):
        super().__init__(aggr=aggr)
        self.mlp = nn.Sequential(nn.Linear(2*in_c, out_c), nn.BatchNorm1d(out_c), nn.ReLU(),
                                 nn.Linear(out_c, out_c),   nn.BatchNorm1d(out_c), nn.ReLU())
    def forward(self, x, edge_index): return self.propagate(edge_index, x=x)
    def message(self, x_i, x_j):      return self.mlp(torch.cat([x_i, x_j - x_i], dim=-1))

class EdgeConvBlock(nn.Module):
    def __init__(self, in_c, out_c, aggr="mean"):
        super().__init__()
        self.conv = EdgeConv(in_c, out_c, aggr)
        self.shortcut = nn.Linear(in_c, out_c); self.act = nn.ReLU()
    def forward(self, x, edge_index): return self.act(self.conv(x, edge_index) + self.shortcut(x))

In [ ]:
# --- CONFIGURABLE ParticleNet: exercises 1 (k), 2 (dynamic), 3 (aggr) are all constructor flags ----
class ParticleNet(nn.Module):
    def __init__(self, in_feat, n_classes=2, k=8, conv_channels=(32,64,64), fc=128,
                 dropout=0.1, aggr="mean", dynamic=True):
        super().__init__()
        self.k, self.dynamic = k, dynamic
        chans = [in_feat, *conv_channels]
        self.blocks = nn.ModuleList([EdgeConvBlock(chans[i], chans[i+1], aggr)
                                     for i in range(len(conv_channels))])
        self.head = nn.Sequential(nn.Linear(conv_channels[-1], fc), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(fc, n_classes))
        self.last_edges = []
    def forward(self, data):
        batch = data.batch if getattr(data, "batch", None) is not None \
                else torch.zeros(data.x.size(0), dtype=torch.long, device=data.x.device)
        feats, coords = data.x, data.pos
        self.last_edges = []
        for block in self.blocks:
            edge_index = knn_graph_native(coords, self.k, batch)   # block0: eta-phi; later: learned (if dynamic)
            self.last_edges.append(edge_index)
            feats = block(feats, edge_index)
            if self.dynamic:                                       # static => keep eta-phi graph every block
                coords = feats
        return self.head(global_mean_pool(feats, batch))

# --- PFN over a PyG batch = DeepSets: sum-pool phi(x) per graph (exercise 6 baseline) ----
class PFN(nn.Module):
    def __init__(self, in_feat, n_classes=2, hidden=(64,128,128), latent=128):
        super().__init__()
        dims = [in_feat, *hidden]; layers = []
        for a, b in zip(dims[:-1], dims[1:]): layers += [nn.Linear(a, b), nn.ReLU()]
        self.phi = nn.Sequential(*layers, nn.Linear(hidden[-1], latent))
        self.head = nn.Sequential(nn.Linear(latent, 128), nn.ReLU(), nn.Linear(128, n_classes))
    def forward(self, data):
        batch = data.batch if getattr(data, "batch", None) is not None \
                else torch.zeros(data.x.size(0), dtype=torch.long, device=data.x.device)
        return self.head(global_add_pool(self.phi(data.x), batch))

# --- GravNet layer (from Module 2) ------------------------------------------------------
class GravNetLayer(nn.Module):
    def __init__(self, in_c, out_c, space_dim=4, prop_dim=22, k=8):
        super().__init__(); self.k = k
        self.lin_s = nn.Linear(in_c, space_dim)
        self.lin_f = nn.Linear(in_c, prop_dim)
        self.lin_out = nn.Sequential(nn.Linear(in_c + 2*prop_dim, out_c), nn.ReLU())
    def forward(self, x, batch):
        s, f = self.lin_s(x), self.lin_f(x)
        j, i = knn_graph_native(s, self.k, batch)
        d2 = ((s[i] - s[j])**2).sum(-1, keepdim=True); w = torch.exp(-d2); msg = f[j]*w
        a_mean = scatter(msg, i, dim=0, dim_size=x.size(0), reduce="mean")
        a_max  = scatter(msg, i, dim=0, dim_size=x.size(0), reduce="max")
        return self.lin_out(torch.cat([x, a_mean, a_max], dim=-1))

In [ ]:
# --- Load JetClass, build PyG Data per jet (top vs QCD) ---------------------------------
import uproot, awkward as ak
CANDIDATE_PATHS = [
    "../data/JetClass_example_100k.root",
    "/Users/sanmay/Documents/ML_SCHOOLS/MLSCHOOL_2023_ICTS/Main_School/JetDataset/JetClass_example_100k.root",
    "./JetClass_example_100k.root",
]
root_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if root_path is None: raise FileNotFoundError("JetClass_example_100k.root not found")
print("Using:", root_path)

MAX_PART  = 128
N_PER_CLS = 2000     # per class; reduced from the module's 6000 so this solutions notebook runs on a laptop
EPOCHS    = 6

tree = uproot.open(root_path)["tree"]
br = tree.arrays(["part_px","part_py","part_pz","part_energy","part_deta","part_dphi",
                  "label_QCD","label_Tbqq","label_Tbl"])
is_qcd = ak.to_numpy(br["label_QCD"]).astype(bool)
is_top = ak.to_numpy(br["label_Tbqq"]).astype(bool) | ak.to_numpy(br["label_Tbl"]).astype(bool)
sel = np.concatenate([np.where(is_qcd)[0][:N_PER_CLS], np.where(is_top)[0][:N_PER_CLS]])
labels = np.concatenate([np.zeros(N_PER_CLS), np.ones(N_PER_CLS)]).astype(np.int64)

px,py,pz,e = br["part_px"][sel], br["part_py"][sel], br["part_pz"][sel], br["part_energy"][sel]
deta,dphi  = br["part_deta"][sel], br["part_dphi"][sel]
pt = np.sqrt(px**2+py**2); dR = np.sqrt(deta**2+dphi**2)
sumpt, sume = ak.sum(pt, axis=1), ak.sum(e, axis=1)
FEATURE_NAMES = ["deta","dphi","log_pt","log_e","log_pt_rel","log_e_rel","dR"]
feat_list = [deta, dphi, np.log(pt+1e-8), np.log(e+1e-8),
             np.log(pt/sumpt+1e-8), np.log(e/sume+1e-8), dR]
def pad(f): return ak.to_numpy(ak.fill_none(ak.pad_none(f, MAX_PART, clip=True), 0.0))
X   = np.stack([pad(f) for f in feat_list], axis=-1).astype(np.float32)
POS = np.stack([pad(deta), pad(dphi)], axis=-1).astype(np.float32)
nparts = np.minimum(ak.to_numpy(ak.num(pt, axis=1)), MAX_PART)

realmask = (np.arange(MAX_PART)[None,:] < nparts[:,None]); flat = X[realmask]
mean, std = flat.mean(0), flat.std(0)+1e-6; Xs = (X-mean)/std

data_list = []
for i in range(len(labels)):
    n = int(nparts[i])
    if n < 2: continue
    data_list.append(Data(x=torch.from_numpy(Xs[i,:n]), pos=torch.from_numpy(POS[i,:n]),
                          y=torch.tensor([int(labels[i])], dtype=torch.long)))
random.shuffle(data_list)
n = len(data_list); a, b = int(0.70*n), int(0.85*n)
splits = {"train": data_list[:a], "val": data_list[a:b], "test": data_list[b:]}
loaders = {k: DataLoader(v, batch_size=64, shuffle=(k=="train")) for k, v in splits.items()}
print("jets:", {k: len(v) for k, v in splits.items()})

In [ ]:
# --- shared train / eval helpers --------------------------------------------------------
@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps, ls, nt = [], [], 0.0, 0
    for data in loader:
        data = data.to(device); logits = model(data)
        ls += F.cross_entropy(logits, data.y, reduction="sum").item(); nt += data.num_graphs
        ys.append(data.y.cpu()); ps.append(F.softmax(logits, -1)[:, 1].cpu())
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return {"loss": ls/nt, "acc": ((p > 0.5).astype(int) == y).mean(),
            "auc": roc_auc_score(y, p), "y": y, "p": p}

def background_rejection(y, p, eff=0.5):
    fpr, tpr, _ = roc_curve(y, p)
    return 1.0 / max(np.interp(eff, tpr, fpr), 1e-12)

def train(model, loaders, epochs=EPOCHS, lr=1e-3, quiet=True):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        for data in loaders["train"]:
            data = data.to(device)
            loss = F.cross_entropy(model(data), data.y)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        if not quiet:
            va = evaluate(model, loaders["val"]); print(f"  epoch {ep+1:2d}: val AUC {va['auc']:.4f}")
    return model, time.time()-t0

def n_params(m): return sum(p.numel() for p in m.parameters())
print("Setup complete.")

## Exercise 1 — k matters

> *Re-train ParticleNet with `k = 4, 8, 16`. How do AUC and background rejection scale with the
> neighbourhood size, and what does it cost in time?*

**The concept.** `k` sets how many neighbours each particle listens to per block. Small `k` = very local,
cheap, but may miss cross-jet structure; large `k` = each particle sees more of the jet (closer to
fully-connected/attention), richer but more edges to process (cost grows ~linearly in `k` for the message
MLP, on top of the fixed `O(N²)` distance computation). We train the identical architecture at three values
of `k` and read off AUC, background rejection `1/ε_B`, and wall-clock time.

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

## Exercise 2 — Static vs dynamic graph

> *Disable the dynamic graph (use the η–φ graph in every block by setting `coords = data.pos` each
> iteration). How much does the dynamic graph actually buy?*

**The concept.** ParticleNet's signature trick is **dynamic** graph construction: block 0 connects angular
neighbours in (η, φ), but every later block rebuilds k-NN in the *learned feature space*, so particles that
are far apart in the detector yet play a similar role in the jet become neighbours. A **static** model keeps
the (η, φ) graph in every block. Our configurable `ParticleNet(dynamic=False)` flips exactly this one switch;
everything else (depth, width, k, parameter count) is identical, so the comparison is clean.

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

## Exercise 3 — Aggregator: mean vs max vs sum

> *Switch EdgeConv's `aggr` from `"mean"` to `"max"` (DGCNN's original choice) and to `"sum"`. Which wins
> for jets, and can you relate it to the Deep Sets discussion in Module 1?*

**The concept.** The aggregator is the **permutation-invariant pool over each node's k neighbours** — the
Deep Sets atom, applied locally. Here is the subtlety versus Module 1: in Module 1 we pooled over the
*whole jet*, whose size `N` varies, so `sum` (which encodes `N`) and `mean` (which forgets it) genuinely
differ. Here we pool over a **fixed `k`** neighbours, so `sum = k·mean` — proportional, and the constant is
absorbed by the following BatchNorm/linear. We therefore expect `sum` and `mean` to behave *almost the same*,
while **`max`** is qualitatively different (it reports the single most extreme neighbour, ignoring the rest).

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

## Exercise 4 — Explicit edge features (ΔR)

> *Add the edge length ΔR_ij as an explicit input to `message()`. Does telling the network the geometry
> help, or does EdgeConv already capture it via x_j − x_i?*

**The concept.** EdgeConv already feeds `x_j − x_i` into the message MLP, and two of our node features are
(Δη, Δφ), so the displacement in angle is *in principle* recoverable. But a network might use an explicit,
rotation-invariant scalar `ΔR_ij = √(Δη² + Δφ²)` more easily than it reconstructs it from a difference of
standardized coordinates. We build `EdgeConvGeo`, which additionally propagates the **physical** (η, φ)
position and appends `ΔR_ij` (computed from the *original* angles, not the learned feature space) to each
message.

In [ ]:
class EdgeConvGeo(MessagePassing):
    """EdgeConv + explicit edge length: message = MLP([x_i, x_j - x_i, dR_ij]),  dR from physical (eta,phi)."""
    def __init__(self, in_c, out_c, aggr="mean"):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def forward(self, x, pos, edge_index):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def message(self, x_i, x_j, pos_i, pos_j):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
class ParticleNetGeo(nn.Module):
    """Dynamic ParticleNet whose EdgeConvGeo blocks also see the physical ΔR of each edge."""
    def __init__(self, in_feat, n_classes=2, k=8, conv_channels=(32,64,64), fc=128, dropout=0.1):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def forward(self, data):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Exercise 5 — Train GravNet

> *Stack 2–3 `GravNetLayer`s + global pool + head and train it. Compare to ParticleNet.*

**The concept.** ParticleNet *recomputes* k-NN from its features; **GravNet** goes one step further and
**learns a low-dimensional coordinate space `S`** whose only job is to define neighbourhoods, then weights
each neighbour's message by a Gaussian of the learned distance `exp(−d²)`. The neighbour *selection* is
non-differentiable, but the *weights* are — so gradients still sculpt `S`. It was designed for irregular
calorimeter geometry where no (η, φ) grid exists. We stack a few layers into a classifier and compare.

In [ ]:
class GravNet(nn.Module):
    def __init__(self, in_feat, n_classes=2, hidden=48, n_layers=3, k=8):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def forward(self, data):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Exercise 6 — PFN vs ParticleNet: where do edges help?

> *Overlay this ROC with Module 1's PFN ROC on the same test jets. Where on the curve do edges help most —
> high signal efficiency or high purity?*

**The concept.** The PFN (Module 1) is the **edge-less** special case — it pools `φ(x_i)` over independent
particles. ParticleNet adds **edges**. Because our PFN reads the *same* PyG batches (`global_add_pool(φ(x))`),
we can train it on the identical splits and overlay the ROCs jet-for-jet. The physics question: edges encode
*relations* (prong angles, subjet structure), which matter most for **rejecting the hardest backgrounds** —
so we expect the biggest ParticleNet gain at the **high-rejection / high-purity** end, and a smaller gain at
loose working points.

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

## Wrap-up

You've reached the end of the worksheet. If every exercise runs without a `NotImplementedError`, you've implemented them all — compare your results and reasoning with the solutions notebook. The through-line across the course: **every model is constrained message passing** — a choice of relations, a permutation-invariant aggregation, and a baked-in symmetry.